# 第9章　Transformer 入門 — 注意機構（Attention）と LLM の正体

ChatGPT などの大規模言語モデル（LLM）の中身は **Transformer**。その心臓が
**注意機構（self-attention）** です。この章では attention を**自分で実装して直感をつかみ**、
PyTorch の既製部品で Transformer を組み、最後に本物の LLM（Hugging Face）への入口まで案内します。

ゴール：attention の式 `softmax(QKᵀ/√d)V` の意味が分かり、小さな Transformer 分類器を動かせる。

> **使い方**：上から順に `Shift + Enter`。GPU 推奨の章です（Colab：ランタイム→ランタイムのタイプを変更→GPU）。

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 9-1. なぜ Transformer？

- 文章は**単語の並び（系列）**。「それ」が何を指すかは離れた単語に依存することも（長距離依存）。
- 昔の RNN は単語を1つずつ順に処理 → 遅い・遠い依存が苦手。
- **Transformer は系列全体を一度に見て、単語同士の関係を直接計算**。並列計算できて高速、長距離も得意。
- この仕組みを巨大データ＋巨大モデルで学習したのが LLM。

## 9-2. 注意機構（Attention）の直感：Q・K・V

「各単語が、他のどの単語に**注目（attend）**して情報をもらうか」を計算します。検索にたとえると：

- **Query（クエリ Q）**＝ 検索したいこと（今の単語が「何を知りたいか」）
- **Key（キー K）**＝ 各単語の見出し（「私はこういう情報を持つ」）
- **Value（バリュー V）**＝ 各単語の中身（実際に渡す情報）

Q と各 K の**相性（内積）**を計算 → softmax で重み（合計1）にする → その重みで V を**加重平均**。
これで「関係の深い単語の情報ほど多く混ぜる」ができます。式は：

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d}}\right)V$$

$\sqrt{d}$ で割るのは、内積が大きくなりすぎて softmax が尖るのを防ぐため（スケーリング）。

## 9-3. attention を手で実装する
小さなテンソルで、重みが「合計1」になり、V の加重平均が出てくることを確認します。

In [ ]:
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V):
    d = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d ** 0.5)   # (..., L, L) 各単語ペアの相性
    weights = F.softmax(scores, dim=-1)             # 行ごとに合計1
    out = weights @ V                               # 重みで V を加重平均
    return out, weights

torch.manual_seed(0)
L, d = 4, 8                       # 系列長4, 次元8
Q = torch.randn(L, d); K = torch.randn(L, d); V = torch.randn(L, d)
out, w = scaled_dot_product_attention(Q, K, V)
print("attention重み (各行の合計=1):")
print(w.round(decimals=2))
print("行ごとの合計:", w.sum(dim=-1))   # すべて 1
print("出力 shape:", out.shape)         # (4, 8)

## 9-4. PyTorch の既製部品：`nn.MultiheadAttention`

実際は Q・K・V を複数の「ヘッド」に分けて並列に注意を計算します（**マルチヘッド注意**）。
PyTorch に用意があります。`batch_first=True` で形を `(バッチ, 系列長, 次元)` にできて直感的。

In [ ]:
import torch.nn as nn
mha = nn.MultiheadAttention(embed_dim=16, num_heads=4, batch_first=True)
x = torch.randn(2, 5, 16)            # (バッチ2, 系列長5, 次元16)
attn_out, attn_w = mha(x, x, x)      # self-attention: Q=K=V=x
print("出力:", attn_out.shape)        # (2, 5, 16)
print("注意重み:", attn_w.shape)       # (2, 5, 5)

## 9-5. Transformer ブロックと位置情報

1ブロック = **マルチヘッド注意 → 全結合（FFN）**、それぞれに**残差接続＋正規化**。PyTorch では `nn.TransformerEncoderLayer` 1つ。

重要な注意点：**attention は順序を知りません**（単語を入れ替えても結果が同じ）。
そこで各位置に**位置情報（positional encoding/embedding）を足して**「何番目か」を教えます。

In [ ]:
layer = nn.TransformerEncoderLayer(d_model=16, nhead=4, dim_feedforward=64, batch_first=True)
encoder = nn.TransformerEncoder(layer, num_layers=2)   # ブロックを2段重ねる
x = torch.randn(2, 5, 16)
print("Transformer出力:", encoder(x).shape)            # (2, 5, 16)

## 9-6. 小さな Transformer 分類器（エンドツーエンド）

トイ課題：長さ8の数字列（0〜9）に、**「7」が含まれていれば 1、なければ 0** を当てる。
attention が「列のどこかに7があるか」を見つけられるかを試します。

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
N, L, VOCAB = 4000, 8, 10
X = torch.randint(0, VOCAB, (N, L))
y = (X == 7).any(dim=1).long()                 # 7 を含めば 1
loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
print("ラベル1の割合:", y.float().mean().item())

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab=10, d=32, seq_len=8, nhead=4, nlayers=2, nclass=2):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)       # 数字 -> ベクトル
        self.pos = nn.Embedding(seq_len, d)     # 位置 -> ベクトル（学習する位置埋め込み）
        layer = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=64, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, nlayers)
        self.head = nn.Linear(d, nclass)
    def forward(self, x):                       # x: (N, L) 整数列
        L = x.size(1)
        pos = torch.arange(L, device=x.device).unsqueeze(0)   # (1, L)
        h = self.tok(x) + self.pos(pos)         # トークン埋め込み + 位置埋め込み
        h = self.enc(h)                         # (N, L, d)
        h = h.mean(dim=1)                       # 系列方向に平均（プーリング）
        return self.head(h)                     # (N, 2)

model = TinyTransformer()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

for epoch in range(8):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    # 簡易精度
    model.eval()
    with torch.no_grad():
        acc = (model(X).argmax(1) == y).float().mean().item()
    print(f"epoch {epoch+1}: loss={loss.item():.4f}  acc={acc:.3f}")

数 epoch で正解率がほぼ 1.0 になれば、Transformer が「列のどこかに7がある」という
**位置に依存しない関係**を attention で学べたということです。

## 9-7. 本物の LLM へ — Hugging Face（読むだけでOK）

ゼロから巨大 LLM を作るのは大変なので、実務では**学習済みモデル**を使います。
`transformers` ライブラリなら数行。下は例（実行には `pip install transformers` とネット接続が必要）：

```python
# pip install transformers
from transformers import pipeline

# 感情分析（英語）
clf = pipeline("sentiment-analysis")
print(clf("I love learning PyTorch!"))
# -> [{'label': 'POSITIVE', 'score': 0.999...}]

# テキスト生成
gen = pipeline("text-generation", model="gpt2")
print(gen("Once upon a time", max_length=30)[0]["generated_text"])
```

自分のデータでの**ファインチューニング**も、ここまで学んだ「学習ループ＋optimizer」の考え方そのままです。

## 9-8. 次のステップ
- 論文「Attention Is All You Need」(2017) — Transformer の原典
- Andrej Karpathy「Let's build GPT」/ nanoGPT — 文字レベルGPTを一から実装（最高の教材）
- Hugging Face Course（無料）: https://huggingface.co/learn
- PyTorch 公式: `nn.Transformer` チュートリアル

## 演習 9
1. トイ課題を「**列の合計が偶数なら1**」に変えて学習してみよう（位置より全体の集約が要る課題）。
2. `nlayers` や `nhead`、`d` を変えて精度・速度の変化を見よう。
3. 9-3 の `scaled_dot_product_attention` で、特定の単語に注意が集中する入力を自作して重み行列を観察しよう。
4. （応用）Hugging Face の `pipeline` を実際に動かして、和文の感情分析モデルも試そう。

In [ ]:
# ここに自分のコードを書いて実行してみよう
